# 02b — HMM/Viterbi Route Reconstruction

Uses Random Forest candidate probabilities as HMM emissions, with directed road connectivity and GPS displacement as transition scores; Viterbi decodes the most likely candidate sequence. A one-intermediate-arc connector is inserted for short skipped segments, and non-connectable transitions stay possible with a penalty so decoding continues across gaps.

**Requires:** outputs of `02a_Bayesian_Optimization.ipynb` (including `bayesian_run_manifest.json`), the feature/cleaned data, and `src/models/bayesian_hmm.py`.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd

working_directory = Path.cwd().resolve()
root_candidates = [working_directory, *working_directory.parents]
root_candidates += [parent / 'PathSnap' for parent in root_candidates]
project_root = next(
    (candidate for candidate in root_candidates
     if (candidate / 'src' / 'models' / 'bayesian_hmm.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate PathSnap/src/models/bayesian_hmm.py from the notebook working directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.bayesian_hmm import (
    evaluate_hmm_partition,
)

feature_root = project_root / 'data' / 'processed' / 'features'
cleaned_root = project_root / 'data' / 'processed' / 'cleaned'
previous_results = project_root / 'results' / 'candidate_models'
output_root = previous_results / 'bayesian_hmm'
manifest_path = output_root / 'bayesian_run_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'{manifest_path} not found. Run 02a_Bayesian_Optimization.ipynb first.')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
bayesian_validation_model_path = Path(manifest['bayesian_validation_model_path'])
tuned_model_path = Path(manifest['tuned_model_path'])
print(f'Project: {project_root}')
print(f'Validation model: {bayesian_validation_model_path}')
print(f'Tuned (final) model: {tuned_model_path}')


## 3. HMM/Viterbi Validation Diagnostics

This diagnostic uses the best Bayesian trial fitted only on the 64 training trips. HMM transition settings are fixed in advance, not tuned on the test data. Compare independent candidate Top-1 predictions with Viterbi-decoded observation accuracy, route continuity, transition feasibility, and expanded-route continuity.

In [4]:
validation_bundle = joblib.load(bayesian_validation_model_path)
validation_hmm_predictions, validation_route_sequences, validation_hmm_metrics = evaluate_hmm_partition(
    feature_root=feature_root,
    cleaned_root=cleaned_root,
    model_bundle=validation_bundle,
    output_root=output_root,
    partition='train',
    split='validation',
)
display(pd.Series(validation_hmm_metrics, name='validation').to_frame())
display(validation_route_sequences.head(20))

,validation
partition,train
split,validation
trip_count,16
scored_trip_count,15
labeled_observations,15456
candidate_rows,908658
candidate_miss_count,0
candidate_miss_rate,0.0
reliable_candidate_references,15456
top1_observation_accuracy,0.900039


,trip_id,route_index,arc_id,inserted_connector,at_observation_id
0,00000001,0,64973,False,0
1,00000001,1,133948,False,6
2,00000001,2,133949,False,7
3,00000001,3,139208,False,8
4,00000001,4,133950,True,10
5,00000001,5,8970,False,10
6,00000001,6,133951,False,14
7,00000001,7,64139,False,17
8,00000001,8,139211,False,19
9,00000001,9,133952,False,21


## 4. Final HMM/Viterbi Evaluation on Student-Test Trips

This is a one-time evaluation of the refitted tuned model. The route-sequence-suspect trip is still decoded for reconstruction output, but its unreliable labels are excluded from scored metrics. Do not select or retune the classifier or transition settings based on these test results.

In [5]:
tuned_bundle = joblib.load(tuned_model_path)
test_hmm_predictions, test_route_sequences, test_hmm_metrics = evaluate_hmm_partition(
    feature_root=feature_root,
    cleaned_root=cleaned_root,
    model_bundle=tuned_bundle,
    output_root=output_root,
    partition='student_test',
)
display(pd.Series(test_hmm_metrics, name='student_test').to_frame())
display(test_route_sequences.head(20))
print(f'Bayesian search results: {output_root / "bayesian_optimization_results.csv"}')
print(f'Student-test Viterbi predictions: {output_root / "student_test_viterbi_route_sequence.csv"}')

,student_test
partition,student_test
split,None
trip_count,10
scored_trip_count,9
labeled_observations,10496
candidate_rows,553240
candidate_miss_count,0
candidate_miss_rate,0.0
reliable_candidate_references,10496
top1_observation_accuracy,0.896246


,trip_id,route_index,arc_id,inserted_connector,at_observation_id
0,00000000,0,17630,False,0
1,00000000,1,17633,False,3
2,00000000,2,211,False,5
3,00000000,3,17638,False,9
4,00000000,4,17645,False,20
5,00000000,5,17659,False,33
6,00000000,6,17674,False,40
7,00000000,7,208,False,58
8,00000000,8,10764,False,68
9,00000000,9,501,False,77


Bayesian search results: C:\Users\NIkAt\coding\codes\Machine Learning\ML_Hackathon\PathSnap\results\candidate_models\bayesian_hmm\bayesian_optimization_results.csv
Student-test Viterbi predictions: C:\Users\NIkAt\coding\codes\Machine Learning\ML_Hackathon\PathSnap\results\candidate_models\bayesian_hmm\student_test_viterbi_route_sequence.csv


## 5. Export Model Artifacts

Exports the six map-matching model artifacts to `artifacts/map_matching_models/`. Existing verified artifacts are left unchanged; incomplete or modified ones are never overwritten.

In [ ]:
import hashlib

from src.models.export_models import export_all_models

artifact_root = project_root / 'artifacts' / 'map_matching_models'
artifact_manifest_path = artifact_root / 'manifest.json'

if artifact_manifest_path.is_file():
    existing_manifest = json.loads(
        artifact_manifest_path.read_text(encoding='utf-8')
    )
    model_records = existing_manifest.get('models', {})
    artifact_files_valid = len(model_records) == 6 and bool(existing_manifest.get('files')) and all(
        (artifact_root / record['path']).is_file()
        and (artifact_root / record['path']).stat().st_size == record['size_bytes']
        and hashlib.sha256((artifact_root / record['path']).read_bytes()).hexdigest()
        == record['sha256']
        for record in existing_manifest['files']
    ) and all(
        (artifact_root / record['file']).is_file()
        for record in model_records.values()
    )
    if not artifact_files_valid:
        raise RuntimeError(
            f'Existing artifacts are incomplete or changed; refusing to overwrite: {artifact_root}'
        )
    print(
        f'Existing verified model artifacts found; leaving them unchanged: {artifact_root}'
    )
elif artifact_root.exists() and any(artifact_root.iterdir()):
    raise RuntimeError(
        'Artifact directory is non-empty but has no valid manifest; '
        f'refusing to overwrite: {artifact_root}'
    )
else:
    artifact_manifest_path, artifact_manifest = export_all_models(
        project_root,
        artifact_root=artifact_root,
    )
    print(
        f'Generated {len(artifact_manifest["models"])} model artifacts: '
        f'{artifact_manifest_path}'
    )